# 01 extract

Builds every derived artefact the analysis notebooks read: per-network json records, representation npz caches, and 4 ms binarised spike caches. Each output is skipped if it already exists, so the notebook is resumable.

In [ ]:
import sys
sys.path.append("../code"); sys.path.append("../model")
import common as C

In [ ]:
import json, time
import numpy as np
import torch
from data_gen import open_file, sparse_data_generator
from model import RSNN

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# binary spike counts stay exact in fp32 only with tf32 off (tf32 truncates the mantissa to 10 bits)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

# 4-class task: language x parity
LABEL_MAP = {i: int(i >= 10) * 2 + int(i % 2 == 1) for i in range(20)}


def shd_generator(units, times, labels, prms, shuffle, epoch, drop_last):
    if prms.get("rate", 0.0) != 0.0 or prms.get("p_del", 0.0) != 0.0:
        yield from sparse_data_generator(units, times, labels, prms, shuffle=shuffle, epoch=epoch, drop_last=drop_last)
        return
    seed = prms["seed"] + epoch
    batch_size = prms["batch_size"]
    nb_steps = prms["nb_steps"]
    nb_units = prms["nb_inputs"]
    inv_dt = 1.0 / prms["time_step"]
    label_arr = labels if isinstance(labels, np.ndarray) else np.array(labels[:])
    sample_index = np.where(np.isin(label_arr, prms["class_list"]))[0]
    num_samples = len(sample_index)
    n_batches = (num_samples // batch_size) if drop_last else -(-num_samples // batch_size)

    np.random.seed(seed)
    if shuffle:
        np.random.shuffle(sample_index)

    for counter in range(n_batches):
        batch_index = sample_index[batch_size * counter:min(num_samples, batch_size * (counter + 1))]
        actual_bs = len(batch_index)
        t_arrays = [np.round(times[idx] * inv_dt).astype(np.int64) for idx in batch_index]
        u_arrays = [units[idx] for idx in batch_index]
        lengths = np.array([len(a) for a in t_arrays], dtype=np.int64)
        if lengths.sum():
            all_ts = np.concatenate(t_arrays)
            all_us = np.concatenate(u_arrays)
            all_bc = np.repeat(np.arange(actual_bs, dtype=np.int64), lengths)
            valid = all_ts < nb_steps
            all_ts, all_us, all_bc = all_ts[valid], all_us[valid], all_bc[valid]
            index_tensor = torch.from_numpy(np.stack([all_bc, all_ts, all_us]))
            values = torch.ones(all_ts.size, dtype=torch.float32)
            x_batch = torch.sparse_coo_tensor(index_tensor, values,
                                              torch.Size([actual_bs, nb_steps, nb_units])).to_dense()
        else:
            x_batch = torch.zeros(actual_bs, nb_steps, nb_units)
        x_batch.clamp_(max=1.0)
        y_batch = torch.tensor([prms["task_label_map"][int(a)] for a in label_arr[batch_index]], dtype=torch.long)
        yield x_batch, y_batch


def load_net(path):
    ck = torch.load(path, map_location=DEV, weights_only=False)
    prms = dict(ck["prms"])
    prms.update({"dtype": torch.float, "device": DEV, "cuda": DEV.type == "cuda"})
    prms["task_label_map"] = LABEL_MAP
    prms["class_list"] = list(range(20))
    model = RSNN(prms, rec=True).to(DEV)
    model.load_state_dict(ck["model_state_dict"])
    model.eval()
    return ck, prms, model


print("loading SHD test set ...", flush=True)
_u, _t, _l = open_file(str(C.SHD / "shd_test.h5"))
UNITS, TIMES, LABELS = list(_u[:]), list(_t[:]), np.array(_l[:])
_u._v_file.close()
print(f"{len(LABELS)} trials, device = {DEV}")

## per-network records

In [ ]:
M_GRID = [1, 2, 3, 4, 6, 8, 12, 16, 24, 32, 48, 64]
# the 2 ms rung actually trains at 3.259 ms because the decay clamp in model/clipper.py floors beta at 2/e
TAU_ACTUAL = {2.0: 3.259, 3.5: 3.5, 15.0: 15.0, 35.0: 35.0, 200.0: 199.5}


def net_list():
    nets = []
    for s in C.SEEDS:
        nets.append(dict(name=C.net_name(s, "het"), group="learned", arch="het", seed=s, tau_nominal=None))
        nowd = s in C.NOWD_SEEDS
        nets.append(dict(name=C.net_name(s, "hom"), group="learned_nowd" if nowd else "learned",
                         arch="hom_nowd" if nowd else "hom", seed=s, tau_nominal=None))
    for s in C.SEEDS_ORIG:
        nets.append(dict(name=C.net_name(s, "scaled"), group="scaled", arch="scaled", seed=s, tau_nominal=None))
    for tau in C.RUNGS:
        for s in C.SEEDS_ORIG:
            nets.append(dict(name=C.net_name(s, C.RUNG_TAG[tau]), group=f"fixed_{tau:g}ms", arch="fixed",
                             seed=s, tau_nominal=tau))
    nets.append(dict(name=C.net_name(202, "fixed200ms"), group="fixed_200ms", arch="fixed",
                     seed=202, tau_nominal=200.0))
    return nets


def ckpt_by_name(name):
    for sub in ("final_40", "controls"):
        p = C.CKPT / sub / f"{name}.pt"
        if p.exists():
            return p
    raise FileNotFoundError(name)


def eig_stats(Cov):
    ev = np.linalg.eigvalsh(Cov)[::-1].clip(min=0)
    tot = ev.sum()
    if tot <= 0:
        return dict(evr=[0.0] * 64, n90=float("nan"), pr=float("nan"), top5=float("nan"), top10=float("nan"))
    evr = ev / tot
    return dict(
        evr=[float(v) for v in evr],
        n90=int(np.searchsorted(np.cumsum(evr), 0.90) + 1),
        pr=float((ev.sum() ** 2) / np.square(ev).sum()),
        top5=float(evr[:5].sum()),
        top10=float(evr[:10].sum()),
    )


# output membrane at native decays, max over time; loop to T-1, mem starts at 0
def membrane_max(drive, alpha, beta):
    B, T, K = drive.shape
    syn = torch.zeros(B, K, device=drive.device)
    mem = torch.zeros(B, K, device=drive.device)
    best = torch.full((B, K), -1e30, device=drive.device)
    for ti in range(T - 1):
        syn = alpha * syn + drive[:, ti]
        mem = beta * mem + (1 - beta) * syn
        best = torch.maximum(best, mem)
    return best


# 5-fold linear decode, least squares to one-hot, numpy only
def decode_cv(X, y, folds=5, seed=0):
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(y))
    oh = np.eye(4)[y]
    accs = []
    for f in range(folds):
        te = order[f::folds]
        tr = np.setdiff1d(order, te)
        A = np.c_[X[tr], np.ones(len(tr))]
        B = np.c_[X[te], np.ones(len(te))]
        W = np.linalg.lstsq(A, oh[tr], rcond=None)[0]
        accs.append(float(((B @ W).argmax(1) == y[te]).mean()))
    return float(np.mean(accs))

In [ ]:
@torch.no_grad()
def analyse(rec):
    ck, prms, model = load_net(ckpt_by_name(rec["name"]))
    sd = ck["model_state_dict"]
    w_out = sd["network.1.w"].to(DEV).float()
    alpha = float(sd["network.1.alpha"].cpu().numpy().ravel()[0])
    beta_out = float(sd["network.1.beta"].cpu().numpy().ravel()[0])

    H, Y, preds = [], [], []
    for x, y in shd_generator(UNITS, TIMES, LABELS, prms, shuffle=False, epoch=0, drop_last=False):
        recs = model(0, 0, x.to(DEV))
        H.append(recs[0][2])                               # [B,T,64] hidden spikes
        preds.append(recs[-1][1].max(1).values.argmax(1).cpu().numpy())
        Y.append(y.numpy())
    H = torch.cat(H)
    Y = np.concatenate(Y)
    pred = np.concatenate(preds)
    N, T, _ = H.shape
    acc = float((pred == Y).mean())

    rate_np = H.mean(1).cpu().numpy().astype(np.float64)   # [N,64] per-trial rate
    active = int((rate_np.sum(0) > 0).sum())
    mean_rate = float(rate_np.mean())

    # pca on the time-resolved matrix; X^T X on 0/1 data is exact in fp32 (counts << 2^24) with tf32 off
    X = H.reshape(-1, 64)
    mu = X.mean(0)
    C1 = ((X.T @ X) / X.shape[0] - torch.outer(mu, mu)).double().cpu().numpy()
    pca_1ms = eig_stats(C1)
    X4 = H[:, ::4, :].reshape(-1, 64)
    mu4 = X4.mean(0)
    C4 = ((X4.T @ X4) / X4.shape[0] - torch.outer(mu4, mu4)).double().cpu().numpy()
    pca_4ms = eig_stats(C4)
    del X4, C4

    # rank-m reconstruction fed through the network's own output membrane
    evals, evecs = np.linalg.eigh(C1)
    V = torch.from_numpy(np.ascontiguousarray(evecs[:, np.argsort(evals)[::-1]].T)).float().to(DEV)
    Vw = V @ w_out
    mw = mu @ w_out
    pc_curve = []
    for m in M_GRID:
        Am = V[:m].T @ Vw[:m]
        correct = 0
        for b0 in range(0, N, 256):
            drive = (H[b0:b0 + 256] - mu) @ Am + mw
            logits = membrane_max(drive, alpha, beta_out)
            correct += int((logits.argmax(1).cpu().numpy() == Y[b0:b0 + 256]).sum())
        pc_curve.append(correct / N)

    # rank-64 reconstruction is the identity, so it must reproduce the live accuracy
    recon_err = abs(pc_curve[-1] - acc)
    assert recon_err <= 0.005

    gain = float(np.max(np.abs(np.linalg.eigvals(sd["network.0.v"].cpu().numpy()))))
    beta_h = sd["network.0.beta"].cpu().numpy().ravel().astype(np.float64)
    tau = -1.0 / np.log(np.clip(beta_h, 1e-9, 1 - 1e-9))
    lin_dec = decode_cv(rate_np, Y)

    hist = ck.get("history", {}).get("test_acc", [])
    out = dict(rec)
    out.update({
        "acc": acc,
        "acc_ckpt": float(hist[-1]) if isinstance(hist, list) and hist else None,
        "n_trials": int(N),
        "active": active,
        "mean_rate": mean_rate,
        "gain": gain,
        "lin_decode": lin_dec,
        "tau_median_ms": float(np.median(tau)),
        "tau_mean_ms": float(tau.mean()),
        "tau_sd_ms": float(tau.std(ddof=1)) if tau.size > 1 else 0.0,
        "tau_cv": float(tau.std(ddof=1) / tau.mean()) if tau.size > 1 and tau.mean() > 0 else 0.0,
        "tau_min_ms": float(tau.min()),
        "tau_max_ms": float(tau.max()),
        "tau_all_ms": [float(v) for v in tau] if tau.size > 1 else [float(tau[0])] * 64,
        "tau_actual_ms": TAU_ACTUAL.get(rec["tau_nominal"]) if rec["tau_nominal"] else None,
        "n90_1ms": pca_1ms["n90"], "pr_1ms": pca_1ms["pr"],
        "top5_1ms": pca_1ms["top5"], "top10_1ms": pca_1ms["top10"], "evr_1ms": pca_1ms["evr"],
        "n90_4ms": pca_4ms["n90"], "pr_4ms": pca_4ms["pr"],
        "top5_4ms": pca_4ms["top5"], "top10_4ms": pca_4ms["top10"], "evr_4ms": pca_4ms["evr"],
        "M_GRID": M_GRID, "pc_curve": pc_curve,
        "recon_err_at_64": recon_err,
    })
    del H, X, model
    if DEV.type == "cuda":
        torch.cuda.empty_cache()
    return out


C.NETJSON.mkdir(parents=True, exist_ok=True)
nets = net_list()
print(f"{len(nets)} networks")
for i, rec in enumerate(nets, 1):
    f = C.NETJSON / f"{rec['name']}.json"
    if f.exists():
        print(f"[{i}/{len(nets)}] {rec['name']}: cached", flush=True)
        continue
    t0 = time.time()
    out = analyse(rec)
    json.dump(out, open(f, "w"), indent=1)
    print(f"[{i}/{len(nets)}] {rec['name']}: acc={out['acc']:.3f} active={out['active']:>2} "
          f"n90={out['n90_1ms']:>2} PR={out['pr_1ms']:.1f} gain={out['gain']:.2f} "
          f"tau={out['tau_median_ms']:.1f}ms ({time.time()-t0:.0f}s)", flush=True)

## representation caches

In [ ]:
NB20, NB50 = 20, 50


# rate [N,64], binned [N,20,64], hidden [N,50,64], u [N,50] input drive per 20 ms bin, y, pred
@torch.no_grad()
def extract_repr(seed, arch):
    ck, prms, model = load_net(C.ckpt_path(seed, arch))
    nb_steps = int(prms["nb_steps"])
    p20, p50 = nb_steps // NB20, nb_steps // NB50

    rate, binned, hidden, udrive, ys, preds = [], [], [], [], [], []
    for x, y in shd_generator(UNITS, TIMES, LABELS, prms, shuffle=False, epoch=0, drop_last=False):
        x = x.to(DEV)
        recs = model(0, 0, x)
        spk = recs[0][2]
        mem_out = recs[-1][1]
        B = spk.shape[0]
        rate.append(spk.mean(1).cpu().numpy())
        binned.append(spk[:, :NB20 * p20].reshape(B, NB20, p20, 64).mean(2).cpu().numpy())
        hidden.append(spk[:, :NB50 * p50].reshape(B, NB50, p50, 64).mean(2).cpu().numpy())
        udrive.append(x[:, :NB50 * p50].sum(2).reshape(B, NB50, p50).mean(2).cpu().numpy())
        preds.append(mem_out.max(1).values.argmax(1).cpu().numpy())
        ys.append(y.numpy())
    np.savez_compressed(
        C.repr_path(seed, arch),
        rate=np.concatenate(rate).astype(np.float32),
        binned=np.concatenate(binned).astype(np.float32),
        hidden=np.concatenate(hidden).astype(np.float32),
        u=np.concatenate(udrive).astype(np.float32),
        y=np.concatenate(ys).astype(np.int64),
        pred=np.concatenate(preds).astype(np.int64),
    )
    del model
    if DEV.type == "cuda":
        torch.cuda.empty_cache()
    yv = np.concatenate(ys); pv = np.concatenate(preds)
    return len(yv), float((pv == yv).mean())


C.REPR.mkdir(parents=True, exist_ok=True)
for seed in C.SEEDS:
    for arch in ("het", "hom"):
        f = C.repr_path(seed, arch)
        if f.exists():
            print(f"seed {seed} {arch}: cached", flush=True)
            continue
        t0 = time.time()
        n, acc = extract_repr(seed, arch)
        print(f"seed {seed} {arch}: n={n} acc={acc:.3f} ({time.time()-t0:.0f}s)", flush=True)

## 4 ms spike cache

In [ ]:
STRIDE, N_TRIALS = 4, 2048


# first 2048 test trials, binary spikes strided to 4 ms, [64, 512000] uint8
@torch.no_grad()
def build_spk(seed, arch):
    ck, prms, model = load_net(C.ckpt_path(seed, arch))
    nb_steps = int(prms["nb_steps"])
    nb_hidden = int(model.network[0].output_size)
    chunks, n = [], 0
    for x, _y in shd_generator(UNITS, TIMES, LABELS, prms, shuffle=False, epoch=0, drop_last=True):
        spk = model(0, 0, x.to(DEV))[0][2]
        spk = spk[:, ::STRIDE, :].detach().cpu().numpy()
        chunks.append(np.transpose(spk, (2, 0, 1)).reshape(nb_hidden, -1))
        n += spk.shape[0]
        if n >= N_TRIALS:
            break
    spk = np.concatenate(chunks, axis=1)[:, :N_TRIALS * (nb_steps // STRIDE)]
    assert np.all((spk == 0) | (spk == 1))
    np.save(C.spk_path(seed, arch), spk.astype(np.uint8))
    del model
    if DEV.type == "cuda":
        torch.cuda.empty_cache()
    return spk.shape


C.SPK.mkdir(parents=True, exist_ok=True)
for seed in C.SEEDS:
    for arch in ("het", "hom"):
        f = C.spk_path(seed, arch)
        if f.exists():
            print(f"seed {seed} {arch}: cached", flush=True)
            continue
        t0 = time.time()
        shape = build_spk(seed, arch)
        print(f"seed {seed} {arch}: built [{shape[0]},{shape[1]}] ({time.time()-t0:.0f}s)", flush=True)